# ML rate-estimation workbench (prototype)

**Purpose**: can gradient-boosted trees estimate a better Poisson goal rate than any
current rung's hand-specified functional form, given a richer feature set in one
place -- ClubElo (from `03-02`'s cache), this pipeline's own decayed goal/xG rates
(`features/strength.py`), and rest days (`mart_team_match.rest_days`, landed since the
curate layer but never consumed by a feature until now, per `docs/model-card.md`'s own
"known limitations")? XGBoost's job is still to produce a Poisson rate, never a 1X2
classifier -- the season simulator needs a full joint scoreline matrix per fixture, and
only a rate-based model gives that.

**New dependency**: `xgboost`, added to `[dependency-groups] dev` in `pyproject.toml`
for this notebook.

**A design simplification worth stating plainly**: every feature here (attack/defence
rate, Elo level and trend, rest days) is a per-club *snapshot* computed once at fit
time, the same convention every other model already uses (`XGRateModel`'s
`_home_advantage`/`_attack`/`_defence`, `ClubEloModel`'s `_current_elo`). Rest days in
particular *could* vary genuinely per fixture, but `MatchModel.scoreline_matrix(home,
away, max_goals)`'s fixed signature has no way to pass fixture-specific context at
predict time without changing the shared protocol every model relies on -- out of
scope for a prototype. So `own_avg_rest_days` here means "this club's average rest
across the training window," not "days since this specific upcoming fixture."

**Two variants, built in order**:
1. **Residual correction** -- XGBoost learns only what `XGRateModel` (the current
   shipped model) gets wrong, using the same features. Cheap, low overfitting risk,
   answers "does ML add anything at all" first.
2. **Full rate model** -- XGBoost predicts the rate outright. An `ELO_ENABLED` toggle
   lets it run on the full 11-season window (comparable to `XGRateModel`'s real 0.1980
   headline) or, with Elo features on, the `03-02`-style overlap window (comparable to
   `ClubEloModel`'s 0.2089).

Standard prototype, not a standing exception (same footing as `03-02`): stays, moves
into `src/plforecast`, or gets deleted once its question is answered.


In [ ]:
import json
from datetime import date
from pathlib import Path

import numpy as np
import polars as pl
import xgboost as xgb

from plforecast import forecasting, viz
from plforecast.artifacts.schema import Provenance
from plforecast.artifacts.writer import (
    build_forecast_document,
    git_provenance,
    now_utc,
    package_versions,
    snapshot_provenance,
)
from plforecast.config import Settings
from plforecast.entities.clubs import load_club_dimension
from plforecast.evaluate.backtest import run_backtest
from plforecast.evaluate.calibration import calibration_by_outcome
from plforecast.evaluate.metrics import outcome_index
from plforecast.evaluate.report import format_table
from plforecast.features.strength import build_club_strength
from plforecast.models.base import (
    ClubId,
    UnknownClubError,
    assert_no_odds_columns,
    drop_odds_columns,
)
from plforecast.models.xg_rates import XGRateModel
from plforecast.simulate.competition import PREMIER_LEAGUE
from plforecast.simulate.engine import simulate_season
from plforecast.simulate.tiebreak import PremierLeagueTiebreaks
from plforecast.storage.db import connect

REPO_ROOT = Path("../..")
NOTEBOOK_CONFIG = Settings(data_dir=REPO_ROOT / "data")
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
EVALUATION_DIR = REPO_ROOT / "docs" / "evaluation"
CLUBELO_CACHE_DIR = REPO_ROOT / "data" / "clubelo-cache"
COMPARISON_SEASON_FLOOR = "2022/23"  # ClubElo's available history starts ~Sept 2022 (03-02)

## Section 1: feature assembly

Loads `03-02`'s cached ClubElo history (run that notebook first if the cache is
empty), joins `mart_team_match.rest_days` onto `stg_matches`, and builds the per-club
snapshot every model below shares.

In [ ]:
def load_elo_history() -> dict[str, pl.DataFrame]:
    frames = {}
    for path in sorted(CLUBELO_CACHE_DIR.glob("*.json")):
        rows = json.loads(path.read_text())
        if not rows:
            continue
        frames[path.stem] = (
            pl.DataFrame(rows)
            .select(
                pl.col("Date").str.slice(0, 10).str.to_date().alias("date"),
                pl.col("Elo").alias("elo"),
            )
            .sort("date")
        )
    return frames


elo_history = load_elo_history()
if not elo_history:
    raise RuntimeError(
        "no cached ClubElo data found -- run 03-02-clubelo-workbench.ipynb's section 1 first"
    )
print(f"loaded Elo history for {len(elo_history)} clubs")

In [ ]:
conn = connect(NOTEBOOK_CONFIG)
team_match = conn.execute("SELECT match_id, club_id, rest_days FROM mart_team_match").pl()
base_matches = conn.execute(
    "SELECT match_id, season, date, home_club_id, away_club_id, home_goals, away_goals, "
    "home_xg, away_xg, result, benchmark_home_odds, benchmark_draw_odds, "
    "benchmark_away_odds, benchmark_source "
    "FROM stg_matches WHERE season != (SELECT max(season) FROM stg_matches) ORDER BY date"
).pl()
conn.close()

home_rest = team_match.rename({"club_id": "home_club_id", "rest_days": "home_rest_days"})
away_rest = team_match.rename({"club_id": "away_club_id", "rest_days": "away_rest_days"})
evaluation_matches = base_matches.join(home_rest, on=["match_id", "home_club_id"], how="left").join(
    away_rest, on=["match_id", "away_club_id"], how="left"
)
print(f"{evaluation_matches.height} matches, {evaluation_matches['season'].n_unique()} seasons")
evaluation_matches.select("home_rest_days", "away_rest_days").describe()

### The shared per-club snapshot and long-format feature frame

Every model below calls these two functions from inside its own `fit(matches)` --
`matches` is already the walk-forward-correct training frame the shared harness
passes in, so nothing here needs its own point-in-time logic beyond `as_of =
matches["date"].max()`, the same convention `ClubEloModel`/`XGRateModel` already use.

In [ ]:
FLOOR = 0.05


def _elo_at(
    history: pl.DataFrame, as_of: date, *, trend_window: int = 5
) -> tuple[float | None, float | None]:
    at_or_before = history.filter(pl.col("date") <= as_of)
    if at_or_before.height == 0:
        return None, None
    level = float(at_or_before[-1, "elo"])
    recent = at_or_before.tail(trend_window)
    if recent.height < 2:
        return level, 0.0
    days = (recent["date"] - recent["date"].min()).dt.total_days().to_numpy().astype(float)
    trend = float(np.polyfit(days, recent["elo"].to_numpy(), 1)[0])
    return level, trend


def build_club_snapshot(
    matches: pl.DataFrame,
    *,
    as_of: date,
    strength_xi: float,
    elo_history: dict[str, pl.DataFrame] | None,
) -> dict[str, dict[str, float | None]]:
    strength = build_club_strength(matches, as_of=as_of, xi=strength_xi)
    snapshot: dict[str, dict[str, float | None]] = {
        row["club_id"]: {
            "home_attack": row["home_attack_rate"],
            "home_defence": row["home_defence_rate"],
            "away_attack": row["away_attack_rate"],
            "away_defence": row["away_defence_rate"],
        }
        for row in strength.iter_rows(named=True)
    }
    rest = (
        pl.concat(
            [
                matches.select(
                    pl.col("home_club_id").alias("club_id"),
                    pl.col("home_rest_days").alias("rest_days"),
                ),
                matches.select(
                    pl.col("away_club_id").alias("club_id"),
                    pl.col("away_rest_days").alias("rest_days"),
                ),
            ]
        )
        .drop_nulls("rest_days")
        .group_by("club_id")
        .agg(pl.col("rest_days").mean().alias("avg_rest_days"))
    )
    for row in rest.iter_rows(named=True):
        snapshot.setdefault(row["club_id"], {})["avg_rest_days"] = row["avg_rest_days"]
    if elo_history:
        for club_id, history in elo_history.items():
            level, trend = _elo_at(history, as_of)
            if level is not None:
                snapshot.setdefault(club_id, {})["elo_level"] = level
                snapshot.setdefault(club_id, {})["elo_trend"] = trend
    return snapshot


FEATURE_COLUMNS_BASE = [
    "is_home",
    "own_attack",
    "own_defence",
    "opp_attack",
    "opp_defence",
    "own_avg_rest_days",
    "opp_avg_rest_days",
]
FEATURE_COLUMNS_ELO = ["elo_diff", "own_elo_trend"]


def _feature_row(snapshot: dict, club: str, opp: str, *, is_home: bool, elo_enabled: bool) -> dict:
    own = snapshot.get(club, {})
    theirs = snapshot.get(opp, {})
    venue, opp_venue = ("home", "away") if is_home else ("away", "home")
    row = {
        "is_home": float(is_home),
        "own_attack": own.get(f"{venue}_attack"),
        "own_defence": own.get(f"{venue}_defence"),
        "opp_attack": theirs.get(f"{opp_venue}_attack"),
        "opp_defence": theirs.get(f"{opp_venue}_defence"),
        "own_avg_rest_days": own.get("avg_rest_days"),
        "opp_avg_rest_days": theirs.get("avg_rest_days"),
    }
    if elo_enabled:
        own_elo, opp_elo = own.get("elo_level"), theirs.get("elo_level")
        row["elo_diff"] = (
            (own_elo - opp_elo) if own_elo is not None and opp_elo is not None else None
        )
        row["own_elo_trend"] = own.get("elo_trend")
    return row


def build_feature_rows(
    matches: pl.DataFrame, snapshot: dict, *, elo_enabled: bool
) -> tuple[pl.DataFrame, np.ndarray]:
    columns = FEATURE_COLUMNS_BASE + (FEATURE_COLUMNS_ELO if elo_enabled else [])
    rows, targets = [], []
    for row in matches.iter_rows(named=True):
        for side, opp_side, is_home in (("home", "away", True), ("away", "home", False)):
            rows.append(
                _feature_row(
                    snapshot,
                    row[f"{side}_club_id"],
                    row[f"{opp_side}_club_id"],
                    is_home=is_home,
                    elo_enabled=elo_enabled,
                )
            )
            targets.append(max(float(row[f"{side}_goals"]), FLOOR))
    frame = pl.DataFrame(rows, schema={c: pl.Float64 for c in columns})
    return frame.select(columns), np.log(np.array(targets))


def to_xgb_matrix(frame: pl.DataFrame) -> np.ndarray:
    """polars nulls -> NaN, which xgboost's tree splits handle natively as missing."""
    return frame.to_numpy().astype(np.float64)

## Section 2: residual correction

`ResidualCorrectedModel` fits `XGRateModel` (the current shipped model) as a baseline,
then a small XGBoost regressor on `log(actual goals) - log(baseline rate)` using the
same feature set. The correction is shrunk by `CORRECTION_WEIGHT` so a noisy
correction can't dominate a reasonable baseline.

In [ ]:
class ResidualCorrectedModel:
    def __init__(
        self,
        *,
        elo_history: dict[str, pl.DataFrame] | None = None,
        elo_enabled: bool = False,
        correction_weight: float = 0.5,
        strength_xi: float = 0.0018,
        max_depth: int = 2,
        n_estimators: int = 100,
        learning_rate: float = 0.05,
        seed: int = 0,
        baseline_xi: float = 0.005,
        baseline_blend: float = 0.75,
        baseline_rho: float = 0.0,
    ) -> None:
        self.elo_history = elo_history or {}
        self.elo_enabled = elo_enabled and bool(self.elo_history)
        self.correction_weight = correction_weight
        self.strength_xi = strength_xi
        self.max_depth = max_depth
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.seed = seed
        self.baseline_xi = baseline_xi
        self.baseline_blend = baseline_blend
        self.baseline_rho = baseline_rho
        self._baseline: XGRateModel | None = None
        self._correction: xgb.XGBRegressor | None = None
        self._snapshot: dict = {}
        self._feature_columns: list[str] = []

    def fit(self, matches: pl.DataFrame) -> "ResidualCorrectedModel":
        assert_no_odds_columns(matches)
        self._baseline = XGRateModel(
            xi=self.baseline_xi, blend=self.baseline_blend, rho=self.baseline_rho
        ).fit(matches)
        as_of = matches["date"].max()
        self._snapshot = build_club_snapshot(
            matches,
            as_of=as_of,
            strength_xi=self.strength_xi,
            elo_history=self.elo_history if self.elo_enabled else None,
        )
        columns = FEATURE_COLUMNS_BASE + (FEATURE_COLUMNS_ELO if self.elo_enabled else [])
        self._feature_columns = columns
        rows, residuals = [], []
        for row in matches.iter_rows(named=True):
            try:
                lam_home, lam_away = self._baseline.rates(row["home_club_id"], row["away_club_id"])
            except UnknownClubError:
                continue
            for side, opp_side, is_home, lam in (
                ("home", "away", True, lam_home),
                ("away", "home", False, lam_away),
            ):
                rows.append(
                    _feature_row(
                        self._snapshot,
                        row[f"{side}_club_id"],
                        row[f"{opp_side}_club_id"],
                        is_home=is_home,
                        elo_enabled=self.elo_enabled,
                    )
                )
                actual = max(float(row[f"{side}_goals"]), FLOOR)
                residuals.append(np.log(actual) - np.log(lam))
        frame = pl.DataFrame(rows, schema={c: pl.Float64 for c in columns}).select(columns)
        self._correction = xgb.XGBRegressor(
            max_depth=self.max_depth,
            n_estimators=self.n_estimators,
            learning_rate=self.learning_rate,
            random_state=self.seed,
            objective="reg:squarederror",
        )
        self._correction.fit(to_xgb_matrix(frame), np.array(residuals))
        return self

    def rates(self, home: ClubId, away: ClubId) -> tuple[float, float]:
        lam_home, lam_away = self._baseline.rates(home, away)  # raises UnknownClubError itself
        rows = [
            _feature_row(self._snapshot, home, away, is_home=True, elo_enabled=self.elo_enabled),
            _feature_row(self._snapshot, away, home, is_home=False, elo_enabled=self.elo_enabled),
        ]
        frame = pl.DataFrame(rows, schema={c: pl.Float64 for c in self._feature_columns}).select(
            self._feature_columns
        )
        correction = self._correction.predict(to_xgb_matrix(frame))
        corrected_home = lam_home * float(np.exp(self.correction_weight * correction[0]))
        corrected_away = lam_away * float(np.exp(self.correction_weight * correction[1]))
        return corrected_home, corrected_away

    def scoreline_matrix(self, home: ClubId, away: ClubId, max_goals: int = 10) -> np.ndarray:
        from scipy.stats import poisson

        lam_home, lam_away = self.rates(home, away)
        goals = np.arange(max_goals + 1)
        matrix = np.outer(poisson.pmf(goals, lam_home), poisson.pmf(goals, lam_away))
        return np.asarray(matrix / matrix.sum())

    @property
    def config_hash(self) -> str:
        import hashlib

        config = {
            "model": "residual-corrected-prototype",
            "elo_enabled": self.elo_enabled,
            "correction_weight": self.correction_weight,
        }
        return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()

In [ ]:
CORRECTION_WEIGHT = 0.5
RESIDUAL_ELO_ENABLED = True
MIN_TRAIN_MATCHES = 100
CADENCE = "gameweek"  # far fewer XGBoost refits than "date" cadence -- see header note

residual_window = (
    evaluation_matches.filter(pl.col("season") >= COMPARISON_SEASON_FLOOR)
    if RESIDUAL_ELO_ENABLED
    else evaluation_matches
)

residual_backtest = run_backtest(
    residual_window,
    lambda: ResidualCorrectedModel(
        elo_history=elo_history,
        elo_enabled=RESIDUAL_ELO_ENABLED,
        correction_weight=CORRECTION_WEIGHT,
    ),
    min_train_matches=MIN_TRAIN_MATCHES,
    cadence=CADENCE,
)
xgrate_backtest = run_backtest(
    residual_window,
    lambda: XGRateModel(xi=0.005, blend=0.75, rho=0.0),
    min_train_matches=MIN_TRAIN_MATCHES,
    cadence=CADENCE,
)
print(
    f"residual-corrected scored: {residual_backtest.n_scored}, "
    f"unrateable: {residual_backtest.unrateable.height}"
)
print(
    f"xg-rates scored: {xgrate_backtest.n_scored}, unrateable: {xgrate_backtest.unrateable.height}"
)

In [ ]:
shared_keys = residual_backtest.scores.select(
    "season", "date", "home_club_id", "away_club_id"
).join(
    xgrate_backtest.scores.select("season", "date", "home_club_id", "away_club_id"),
    on=["season", "date", "home_club_id", "away_club_id"],
    how="inner",
)


def _on_rows(scores: pl.DataFrame, keys: pl.DataFrame) -> pl.DataFrame:
    return scores.join(keys, on=["season", "date", "home_club_id", "away_club_id"], how="semi")


def _score_row(name: str, scores: pl.DataFrame, keys: pl.DataFrame) -> dict:
    restricted = _on_rows(scores, keys)
    return {
        "model": name,
        "n": restricted.height,
        "rps": restricted["rps"].mean(),
        "log_loss": restricted["log_loss"].mean(),
        "brier": restricted["brier"].mean(),
    }


print(f"identical rows: {shared_keys.height}")
print(
    format_table(
        [
            _score_row("residual-corrected", residual_backtest.scores, shared_keys),
            _score_row("xg-rates", xgrate_backtest.scores, shared_keys),
        ]
    )
)

## Section 3: full feature-rich rate model

`XGBoostRateModel` predicts the rate outright (no baseline to correct) from the same
feature set, trained on the "long" double-appearance shape `features/strength.py` and
`storage/curate.py`'s `build_team_match` already use -- one row per side per match.

In [ ]:
class XGBoostRateModel:
    def __init__(
        self,
        *,
        elo_history: dict[str, pl.DataFrame] | None = None,
        elo_enabled: bool = False,
        strength_xi: float = 0.0018,
        max_depth: int = 3,
        n_estimators: int = 150,
        learning_rate: float = 0.05,
        seed: int = 0,
    ) -> None:
        self.elo_history = elo_history or {}
        self.elo_enabled = elo_enabled and bool(self.elo_history)
        self.strength_xi = strength_xi
        self.max_depth = max_depth
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.seed = seed
        self._model: xgb.XGBRegressor | None = None
        self._snapshot: dict = {}
        self._feature_columns: list[str] = []

    def fit(self, matches: pl.DataFrame) -> "XGBoostRateModel":
        assert_no_odds_columns(matches)
        as_of = matches["date"].max()
        self._snapshot = build_club_snapshot(
            matches,
            as_of=as_of,
            strength_xi=self.strength_xi,
            elo_history=self.elo_history if self.elo_enabled else None,
        )
        frame, target = build_feature_rows(matches, self._snapshot, elo_enabled=self.elo_enabled)
        self._feature_columns = frame.columns
        self._model = xgb.XGBRegressor(
            max_depth=self.max_depth,
            n_estimators=self.n_estimators,
            learning_rate=self.learning_rate,
            random_state=self.seed,
            objective="reg:squarederror",
        )
        self._model.fit(to_xgb_matrix(frame), target)
        return self

    def rates(self, home: ClubId, away: ClubId) -> tuple[float, float]:
        known = [
            c
            for c in (home, away)
            if c not in self._snapshot or self._snapshot[c].get("home_attack") is None
        ]
        if known:
            raise UnknownClubError(f"club(s) with no snapshot: {known!r}")
        rows = [
            _feature_row(self._snapshot, home, away, is_home=True, elo_enabled=self.elo_enabled),
            _feature_row(self._snapshot, away, home, is_home=False, elo_enabled=self.elo_enabled),
        ]
        frame = pl.DataFrame(rows, schema={c: pl.Float64 for c in self._feature_columns}).select(
            self._feature_columns
        )
        preds = self._model.predict(to_xgb_matrix(frame))
        return float(np.exp(preds[0])), float(np.exp(preds[1]))

    def scoreline_matrix(self, home: ClubId, away: ClubId, max_goals: int = 10) -> np.ndarray:
        from scipy.stats import poisson

        lam_home, lam_away = self.rates(home, away)
        goals = np.arange(max_goals + 1)
        matrix = np.outer(poisson.pmf(goals, lam_home), poisson.pmf(goals, lam_away))
        return np.asarray(matrix / matrix.sum())

    @property
    def config_hash(self) -> str:
        import hashlib

        config = {"model": "xgboost-rate-prototype", "elo_enabled": self.elo_enabled}
        return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()

    @property
    def feature_importances(self) -> dict[str, float]:
        return dict(
            zip(self._feature_columns, self._model.feature_importances_.tolist(), strict=True)
        )

### `ELO_ENABLED = False`: full 11-season window

Directly comparable to `XGRateModel`'s real 0.1980 headline RPS from
`docs/evaluation.md`.

In [ ]:
full_no_elo_backtest = run_backtest(
    evaluation_matches,
    lambda: XGBoostRateModel(elo_enabled=False),
    min_train_matches=MIN_TRAIN_MATCHES,
    cadence=CADENCE,
)
xgrate_full_backtest = run_backtest(
    evaluation_matches,
    lambda: XGRateModel(xi=0.005, blend=0.75, rho=0.0),
    min_train_matches=MIN_TRAIN_MATCHES,
    cadence=CADENCE,
)
full_keys = full_no_elo_backtest.scores.select(
    "season", "date", "home_club_id", "away_club_id"
).join(
    xgrate_full_backtest.scores.select("season", "date", "home_club_id", "away_club_id"),
    on=["season", "date", "home_club_id", "away_club_id"],
    how="inner",
)
print(f"identical rows (full window): {full_keys.height}")
print(
    format_table(
        [
            _score_row("xgboost-no-elo", full_no_elo_backtest.scores, full_keys),
            _score_row("xg-rates (full window)", xgrate_full_backtest.scores, full_keys),
        ]
    )
)

### `ELO_ENABLED = True`: overlap window

Restricted to `COMPARISON_SEASON_FLOOR` onward, same restriction `03-02` used for
`ClubEloModel` (RPS 0.2089 there).

In [ ]:
elo_window = evaluation_matches.filter(pl.col("season") >= COMPARISON_SEASON_FLOOR)
full_with_elo_backtest = run_backtest(
    elo_window,
    lambda: XGBoostRateModel(elo_history=elo_history, elo_enabled=True),
    min_train_matches=MIN_TRAIN_MATCHES,
    cadence=CADENCE,
)
xgrate_overlap_backtest = run_backtest(
    elo_window,
    lambda: XGRateModel(xi=0.005, blend=0.75, rho=0.0),
    min_train_matches=MIN_TRAIN_MATCHES,
    cadence=CADENCE,
)
overlap_keys = full_with_elo_backtest.scores.select(
    "season", "date", "home_club_id", "away_club_id"
).join(
    xgrate_overlap_backtest.scores.select("season", "date", "home_club_id", "away_club_id"),
    on=["season", "date", "home_club_id", "away_club_id"],
    how="inner",
)
print(f"identical rows (overlap window): {overlap_keys.height}")
print(
    format_table(
        [
            _score_row("xgboost-with-elo", full_with_elo_backtest.scores, overlap_keys),
            _score_row("xg-rates (overlap window)", xgrate_overlap_backtest.scores, overlap_keys),
        ]
    )
)
print("reference from 03-02, same overlap window: clubelo-prototype 0.2089, xg-rates 0.1990")

### Feature importance

Fit once more on the full window to inspect which features the trees actually split on.

In [ ]:
importance_model = XGBoostRateModel(elo_enabled=False).fit(drop_odds_columns(evaluation_matches))
importance_df = pl.DataFrame(
    [{"feature": k, "importance": v} for k, v in importance_model.feature_importances.items()]
).sort("importance", descending=True)
importance_df

In [ ]:
importance_model_elo = XGBoostRateModel(elo_history=elo_history, elo_enabled=True).fit(
    drop_odds_columns(elo_window)
)
importance_df_elo = pl.DataFrame(
    [{"feature": k, "importance": v} for k, v in importance_model_elo.feature_importances.items()]
).sort("importance", descending=True)
importance_df_elo

Calibration for whichever variant looks best -- edit `BEST_MODEL` after reviewing the tables above.

In [ ]:
BEST_MODEL = "xgboost-no-elo"  # "residual-corrected" | "xgboost-no-elo" | "xgboost-with-elo"

_by_name = {
    "residual-corrected": (residual_backtest, shared_keys),
    "xgboost-no-elo": (full_no_elo_backtest, full_keys),
    "xgboost-with-elo": (full_with_elo_backtest, overlap_keys),
}
best_backtest, best_keys = _by_name[BEST_MODEL]
best_shared = _on_rows(best_backtest.scores, best_keys)
probs = best_shared.select("p_home", "p_draw", "p_away").to_numpy()
outcomes = outcome_index(best_shared["result"].to_list())
calibration_rows = calibration_by_outcome(probs, outcomes).to_dicts()
viz.calibration_chart(calibration_rows)

## Section 4: forecast visualization

A live forecast from `BEST_MODEL`, fit on every available match, the same
final-table/position-matrix/points-ridgeline treatment as `03-01`/`03-02`, compared
against `XGRateModel`'s current forecast.

In [ ]:
conn = connect(NOTEBOOK_CONFIG)
fixtures = conn.execute(
    "SELECT fixture_id, season, gameweek, kickoff_time, home_club_id, away_club_id, "
    "home_goals, away_goals, finished FROM stg_fixtures ORDER BY kickoff_time, fixture_id"
).pl()
# unlike evaluation_matches (deliberately excludes the in-progress season for backtest
# fairness), the live forecast needs this season's own played matches too, or a
# genuinely blank promoted club (Coventry, Hull) has no snapshot at all -- the same
# gap forecasting.fit_and_simulate's promoted-club prior exists to fill, which none of
# this notebook's prototype models replicate. Known simplification, not fixed here.
full_history_team_match = conn.execute(
    "SELECT match_id, club_id, rest_days FROM mart_team_match"
).pl()
full_history_base = conn.execute(
    "SELECT match_id, season, date, home_club_id, away_club_id, home_goals, away_goals, "
    "home_xg, away_xg, result FROM stg_matches ORDER BY date"
).pl()
conn.close()

full_history_home_rest = full_history_team_match.rename(
    {"club_id": "home_club_id", "rest_days": "home_rest_days"}
)
full_history_away_rest = full_history_team_match.rename(
    {"club_id": "away_club_id", "rest_days": "away_rest_days"}
)
full_history = full_history_base.join(
    full_history_home_rest, on=["match_id", "home_club_id"], how="left"
).join(full_history_away_rest, on=["match_id", "away_club_id"], how="left")

season_label = str(fixtures["season"][0])
played = fixtures.filter(pl.col("finished"))
remaining = fixtures.filter(~pl.col("finished"))

live_training = (
    full_history.filter(pl.col("season") >= COMPARISON_SEASON_FLOOR)
    if BEST_MODEL == "xgboost-with-elo"
    else full_history
)
if BEST_MODEL == "residual-corrected":
    live_fitted = ResidualCorrectedModel(
        elo_history=elo_history,
        elo_enabled=RESIDUAL_ELO_ENABLED,
        correction_weight=CORRECTION_WEIGHT,
    ).fit(live_training)
elif BEST_MODEL == "xgboost-with-elo":
    live_fitted = XGBoostRateModel(elo_history=elo_history, elo_enabled=True).fit(live_training)
else:
    live_fitted = XGBoostRateModel(elo_enabled=False).fit(live_training)

live_clubs = set(live_fitted._snapshot)
ratable_remaining = remaining.filter(
    pl.col("home_club_id").is_in(list(live_clubs)) & pl.col("away_club_id").is_in(list(live_clubs))
)
n_excluded = remaining.height - ratable_remaining.height
print(f"fixtures excluded (club missing from {BEST_MODEL}'s snapshot): {n_excluded}")

In [ ]:
SIMULATIONS = 5_000
SEED = 42

ml_result = simulate_season(
    played.select("home_club_id", "away_club_id", "home_goals", "away_goals"),
    ratable_remaining.select("home_club_id", "away_club_id"),
    live_fitted,
    PremierLeagueTiebreaks(PREMIER_LEAGUE),
    n_simulations=SIMULATIONS,
    max_goals=10,
    seed=SEED,
)
assert (ml_result.position_counts.sum(axis=1) == SIMULATIONS).all()
assert (ml_result.position_counts.sum(axis=0) == SIMULATIONS).all()
print(f"simulated {len(ml_result.club_ids)} clubs, {SIMULATIONS:,} simulations")

### Final projected table

In [ ]:
dimension = load_club_dimension()
display_names = dict(
    zip(
        dimension.frame["club_id"].to_list(), dimension.frame["display_name"].to_list(), strict=True
    )
)

generated_at = now_utc()
sha, dirty = git_provenance(REPO_ROOT)
provenance = Provenance(
    git_sha=sha,
    git_dirty=dirty,
    model=BEST_MODEL,
    model_config_hash=live_fitted.config_hash,
    simulations=SIMULATIONS,
    random_seed=SEED,
    sources=snapshot_provenance(NOTEBOOK_CONFIG.raw_dir),
    package_versions=package_versions(),
)
ml_forecast_doc = build_forecast_document(
    ml_result,
    played=played,
    display_names=display_names,
    season=season_label.replace("/", "-"),
    as_of_gameweek=int(played["gameweek"].max() or 0),
    generated_at=generated_at,
    provenance=provenance,
)
ml_final_table = pl.DataFrame(
    [
        {
            "pos": i + 1,
            "club": club.display_name,
            "pld": club.current.played,
            "pts": club.current.points,
            "E[pts]": round(club.projected.points.mean, 1),
            "title%": round(club.projected.title * 100, 1),
            "top4%": round(club.projected.top_four * 100, 1),
            "rel%": round(club.projected.relegation * 100, 1),
        }
        for i, club in enumerate(ml_forecast_doc.clubs)
    ]
)
ml_final_table

In [ ]:
viz.position_matrix(ml_result.club_ids, ml_result.position_pmf, display_names)

In [ ]:
viz.points_ridgeline(ml_result.club_ids, ml_result.points, display_names)

**A real gap to flag, not to paper over**: none of this notebook's prototype models
have any promoted-club-prior mechanism -- `features/priors.py` is `forecasting
.fit_and_simulate`-only machinery this notebook never calls. A club with almost no
2026/27 history (Coventry) gets rated from whatever handful of matches it has actually
played, the exact failure `docs/model-card.md` documents as the pre-prior bug: an
implausibly harsh forecast from a small sample. Expect this model's promoted-club rows
to look worse than `XGRateModel`'s prior-informed ones for exactly that reason, not
because the underlying rate estimation is worse for those clubs specifically.

### Compare against `XGRateModel`'s current forecast

In [ ]:
xgrate_run = forecasting.fit_and_simulate(
    "xg-rates", simulations=SIMULATIONS, seed=SEED, config=NOTEBOOK_CONFIG
)
xgrate_forecast_doc = build_forecast_document(
    xgrate_run.result,
    played=xgrate_run.played,
    display_names=xgrate_run.display_names,
    season=xgrate_run.season_label.replace("/", "-"),
    as_of_gameweek=xgrate_run.as_of_gameweek,
    generated_at=now_utc(),
    provenance=provenance,
)
comparison = (
    ml_final_table.select("club", pl.col("pos").alias("ml_pos"), pl.col("E[pts]").alias("ml_e_pts"))
    .join(
        pl.DataFrame(
            [
                {
                    "club": c.display_name,
                    "xgrate_pos": i + 1,
                    "xgrate_e_pts": round(c.projected.points.mean, 1),
                }
                for i, c in enumerate(xgrate_forecast_doc.clubs)
            ]
        ),
        on="club",
        how="full",
        coalesce=True,
    )
    .sort("xgrate_pos")
)
comparison